In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from utils import *

# Locate the drop contour using convexity defects.

In [ ]:
#Read the three-channel image and convert it to grayscale.
img1_3ch=cv2.imread('1.jpg')
img1=cv2.cvtColor(img1_3ch,cv2.COLOR_BGR2GRAY)
print('image size: ',img1.shape)

In [ ]:
plt.imshow(img1,cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# Apply Gaussian blur and Canny edge detection.
bl=cv2.GaussianBlur(img1,(9,9),4)
ca=cv2.Canny(bl,50,60)
threshed=cv2.dilate(ca,(15,15),iterations=5)
plt.imshow(threshed,cmap='gray') 
plt.show()

In [ ]:
# Extract continuous contours and select the longest one.
cnts,hier=cv2.findContours(threshed,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
denoised=threshed.copy()
lengths=np.zeros(len(cnts))
for t,c in enumerate(cnts):
    lengths[t]=cv2.arcLength(c,False)
Curve=cnts[np.argmax(lengths)]

In [ ]:
# Display the detected contour.
img_contour=img1_3ch.copy()
img_contour=cv2.drawContours(img_contour,[Curve],-1,(0,255,0),4)

b,g,r=cv2.split(img_contour)
img_contour=cv2.merge([r,g,b])

fig=plt.figure(figsize=(8,6))
plt.imshow(img_contour)
plt.axis('off')
plt.savefig('./CurveDetected')
plt.show()

In [ ]:
#Downsample the dense high-resolution contour to reduce computation.
original_length=len(Curve)
choice=np.linspace(0,original_length,
                   1000 if original_length>1000 else original_length,
                   endpoint=False,dtype=np.int32)


print('Contour points before downsampling:',len(Curve))
Curve=Curve[choice]
print('Contour points after downsampling:',len(Curve))

In [ ]:
#Image y coordinates increase downward, so the maximum y identifies the drop tip.

#The contour is stored counterclockwise: points after the tip lie on the right and points before it on the left.

#PointsNUM=int(Curve.shape[0]*0.4)
tip_index=np.argmax(Curve[::,0,1])
print('Drop tip index:',tip_index)

#Build a convex hull for convexity-defect detection.
hull_points=cv2.convexHull(Curve,returnPoints=True)
hull_indices=cv2.convexHull(Curve,returnPoints=False)
hull_indices[::-1].sort(axis=0)

In [ ]:
convexDefects=cv2.convexityDefects(Curve,hull_indices)
convexDef_length=convexDefects[::,0,1]-convexDefects[::,0,0]
# (convexDef_indices,)=np.where(convexDef_length>50)
convexDef_indices=np.argsort(convexDef_length)[-3:-1]

convexDef_length

In [ ]:
# Select the left and right boundary points manually; this step is not fully automated.
# Inspect the preceding output for the two large, similar defect values marking the boundaries.
# Enter the two manually selected positions here.
a=2 # Second entry.
b=-2 # Original note: third from last; b=-2 actually selects the second-to-last entry.

a = min(a, convexDefects.shape[0] - 1)
b = min(b, convexDefects.shape[0] - 1)
left_index,right_index=convexDefects[[a,b],0,2]

print('Left boundary index: ',left_index)
print('Right boundary index: ',right_index)

In [ ]:
# Display the segmentation result.
img_contour=img1_3ch.copy()
img_contour=cv2.drawContours(img_contour,[Curve],-1,(0,255,0),25)
img_contour=cv2.drawContours(img_contour,[hull_points],-1,(255,40,85),25)
img_contour=cv2.circle(img_contour,Curve[left_index].squeeze(),20,color=(0,80,255),thickness=-1)
img_contour=cv2.circle(img_contour,Curve[right_index].squeeze(),20,color=(0,80,255),thickness=-1)
b,g,r=cv2.split(img_contour)
img_contour=cv2.merge([r,g,b])

fig=plt.figure(figsize=(8,6))
#plt.suptitle('Convex hull and the original curve')
plt.axis('off')
fig.savefig('./ConvexHull_1.jpg')
plt.imshow(img_contour)
plt.show()

In [ ]:
#Extract the pendant-drop segment of the contour.
l,r=sorted([left_index,right_index])
PendantCurve=Curve[l:r+1]

#Compute the capillary diameter from the distance between boundary points.
diameter=np.sum(np.square(PendantCurve[0,0,::]-PendantCurve[-1,0,::]))
diameter=np.sqrt(diameter)
print('Capillary diameter:',diameter,'pixels')

In [ ]:
img_contour=img1_3ch.copy()
img_contour=cv2.drawContours(img_contour,[PendantCurve],-1,(0,255,0),10)
b,g,r=cv2.split(img_contour)
img_contour=cv2.merge([r,g,b])

fig=plt.figure(figsize=(8,6))
#plt.suptitle('The detected silhouette of pendant')
plt.imshow(img_contour)
plt.axis('off')
fig.savefig('./DetectedCurve')
plt.show()

In [ ]:
#Move the drop tip to the origin and reverse the vertical axis.
PendantCurve_numerical=(PendantCurve-PendantCurve[tip_index-left_index]).reshape(-1,2)
PendantCurve_numerical=-np.array(PendantCurve_numerical,dtype=np.float32)

print(PendantCurve_numerical.shape)

In [ ]:
#Final extracted contour points.
from matplotlib import rcParams
fig=plt.figure(figsize=(8,6))
rcParams['font.size']=10
r,z=PendantCurve_numerical.T
r = r[z<=500]
z = z[z<=500]
plt.scatter(r,z,s=2.0)
plt.axis('equal')
plt.xlabel(r'$x$')
plt.ylabel(r'$y$')
fig.savefig('./FinalCurve.jpg')
plt.show()

In [ ]:
curve_data = np.hstack((r.reshape(-1,1), z.reshape(-1,1)))
np.save('curve_data.npy', curve_data)

In [ ]:
# Save the extracted contour data.
import pickle
with open('./Exp1.pkl','wb') as f:
    pickle.dump(PendantCurve_numerical,f)